In [2]:
import sys, torch
print("Colab ortamında mıyız?:", 'google.colab' in sys.modules)
print("GPU (CUDA) aktif mi?:", torch.cuda.is_available())


Colab ortamında mıyız?: True
GPU (CUDA) aktif mi?: False


In [3]:
!pip install -q torch_geometric


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 42.5 MB/s eta 0:00:00


In [4]:
import torch
from torch_geometric.data import Data


def generate_scene_graph():
    num_nodes = torch.randint(4, 9, (1,)).item()

    object_types = torch.empty(num_nodes, dtype=torch.float)
    object_types[0] = 0  # Dron
    object_types[1] = 1  # Ana Hedef
    if num_nodes > 2:
        object_types[2:] = torch.randint(2, 4, (num_nodes - 2,), dtype=torch.long).float()

    positions = torch.rand(num_nodes, 2) * 100
    speeds = torch.rand(num_nodes, 1) * 20
    threat_scores = torch.rand(num_nodes, 1)
    x = torch.cat((object_types.unsqueeze(1), positions, speeds, threat_scores), dim=1)

    distances = torch.cdist(positions, positions)
    edge_mask = (distances < 35) & ~torch.eye(num_nodes, dtype=torch.bool)
    edge_index = edge_mask.nonzero(as_tuple=False).t().contiguous().to(torch.long)

    drone_to_obstacle = (distances[0, object_types == 3] < 25).any()
    target_to_civilian = (distances[1, object_types == 2] < 25).any()
    if drone_to_obstacle:
        label = 2  # Kaçın
    elif target_to_civilian:
        label = 1  # Gözlemle
    else:
        label = 0  # Güvenli Yaklaş

    y = torch.tensor([label], dtype=torch.long)
    return Data(x=x, edge_index=edge_index, y=y)


scene_graph = generate_scene_graph()
print(scene_graph)
print("x shape:", scene_graph.x.shape)
print("edge_index shape:", scene_graph.edge_index.shape)
print("y:", scene_graph.y)


Data(x=[4, 5], edge_index=[2, 4], y=[1])
x shape: torch.Size([4, 5])
edge_index shape: torch.Size([2, 4])
y: tensor([1])


In [5]:
from torch_geometric.loader import DataLoader

train_dataset = [generate_scene_graph() for _ in range(1000)]
test_dataset = [generate_scene_graph() for _ in range(200)]

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Eğitim grafı sayısı: {len(train_dataset)}")
print(f"Test grafı sayısı: {len(test_dataset)}")
print(f"Eğitim batch sayısı: {len(train_loader)}")
print(f"Test batch sayısı: {len(test_loader)}")


Eğitim grafı sayısı: 1000
Test grafı sayısı: 200
Eğitim batch sayısı: 32
Test batch sayısı: 7


In [6]:
import torch.nn as nn
from torch_geometric.nn import GCNConv, global_mean_pool


class TacticalDroneGNN(nn.Module):
    def __init__(self, input_dim=5, hidden_dim=32, num_classes=3):
        super().__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, data):
        x = self.conv1(data.x, data.edge_index)
        x = torch.relu(x)
        x = self.conv2(x, data.edge_index)
        x = torch.relu(x)
        x = global_mean_pool(x, data.batch)
        return self.classifier(x)


model = TacticalDroneGNN()
batch = next(iter(train_loader))
logits = model(batch)
print("Model çıktısı boyutu:", logits.shape)


Model çıktısı boyutu: torch.Size([32, 3])


In [7]:
model = TacticalDroneGNN()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
num_epochs = 30

for epoch in range(1, num_epochs + 1):
    model.train()
    total_loss = 0.0

    for data in train_loader:
        optimizer.zero_grad()
        outputs = model(data)
        loss = criterion(outputs, data.y.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if epoch % 5 == 0:
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for data in test_loader:
                outputs = model(data)
                predictions = outputs.argmax(dim=1)
                labels = data.y.view(-1)
                correct += (predictions == labels).sum().item()
                total += labels.size(0)

        average_loss = total_loss / len(train_loader)
        accuracy = correct / total
        print(
            f"Epoch [{epoch}/{num_epochs}], "
            f"Loss: {average_loss:.4f}, "
            f"Test Accuracy: {accuracy:.4f}"
        )

torch.save(model.state_dict(), "tactical_gnn.pth")
print("Model tactical_gnn.pth olarak kaydedildi.")


Epoch [5/30], Loss: 0.9954, Test Accuracy: 0.5100
Epoch [10/30], Loss: 1.0031, Test Accuracy: 0.5100
Epoch [15/30], Loss: 0.9934, Test Accuracy: 0.5100
Epoch [20/30], Loss: 0.9841, Test Accuracy: 0.5100
Epoch [25/30], Loss: 0.9715, Test Accuracy: 0.5100
Epoch [30/30], Loss: 0.9817, Test Accuracy: 0.5100
Model tactical_gnn.pth olarak kaydedildi.


In [8]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [9]:
!cp tactical_gnn.pth /content/drive/MyDrive/


In [10]:
import os
import shutil
import torch
from google.colab import drive

# 1. Modeli bulunduğumuz geçici dizine kesin olarak kaydet
# (AI asistanının model değişkenini 'model' olarak adlandırdığını varsayıyoruz)
torch.save(model, '/content/tactical_gnn.pth')

if os.path.exists('/content/tactical_gnn.pth'):
    print("1️⃣ Model geçici belleğe başarıyla kaydedildi.")
    
    # 2. Drive'ı bağla (Ekranda çıkan linke tıklayıp izin vermen gerekebilir)
    drive.mount('/content/drive')
    
    # 3. Dosyayı Drive'ın ana dizinine kopyala
    hedef_yol = '/content/drive/MyDrive/tactical_gnn.pth'
    shutil.copy('/content/tactical_gnn.pth', hedef_yol)
    
    print(f"2️⃣ HARİKA! Dosya Drive'a kopyalandı.")
    print("👉 Şimdi tarayıcıdan drive.google.com adresine girip 'En Son' (Recent) sekmesine bakabilirsin.")
else:
    print("❌ Model bulunamadı. Lütfen eğitim döngüsünü (train_loader) çalıştırdığın hücrenin hata vermeden bittiğinden emin ol.")

1️⃣ Model geçici belleğe başarıyla kaydedildi.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
2️⃣ HARİKA! Dosya Drive'a kopyalandı.
👉 Şimdi tarayıcıdan drive.google.com adresine girip 'En Son' (Recent) sekmesine bakabilirsin.


In [ ]:
!pip install -q fastapi uvicorn nest_asyncio pyngrok torch-geometric

import shutil
import threading
from pathlib import Path

import nest_asyncio
import torch
import torch.nn as nn
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from pyngrok import ngrok
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, global_mean_pool
import uvicorn


# Ngrok Dashboard > Your Authtoken adresinden kendi tokeninizi girin.
NGROK_AUTH_TOKEN = "3Kp3O8NxnUGeZFLFqjTT0oZBzCs_5d4cLx7xXT8a2drG8ZnUd"
if NGROK_AUTH_TOKEN == "3Kp3O8NxnUGeZFLFqjTT0oZBzCs_5d4cLx7xXT8a2drG8ZnUd":
    raise ValueError("NGROK_AUTH_TOKEN alanına kendi ngrok tokeninizi yazın.")


MODEL_PATH = Path("/content/tactical_gnn.pth")
DRIVE_MODEL_PATH = Path("/content/drive/MyDrive/tactical_gnn.pth")
if not MODEL_PATH.exists() and DRIVE_MODEL_PATH.exists():
    shutil.copy(DRIVE_MODEL_PATH, MODEL_PATH)
if not MODEL_PATH.exists():
    from google.colab import files
    uploaded_files = files.upload()
    if "tactical_gnn.pth" not in uploaded_files:
        raise FileNotFoundError("tactical_gnn.pth yüklenmedi.")


class TacticalDroneGNN(nn.Module):
    def __init__(self, input_dim=5, hidden_dim=32, num_classes=3):
        super().__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, data):
        x = torch.relu(self.conv1(data.x, data.edge_index))
        x = torch.relu(self.conv2(x, data.edge_index))
        x = global_mean_pool(x, data.batch)
        return self.classifier(x)


model = TacticalDroneGNN()
model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
model.eval()


class SceneObject(BaseModel):
    type: int = Field(..., ge=0, le=3)
    x: float = Field(..., ge=0, le=100)
    y: float = Field(..., ge=0, le=100)
    speed: float = Field(0, ge=0, le=20)
    threat: float = Field(0, ge=0, le=1)


class PredictionRequest(BaseModel):
    objects: list[SceneObject] = Field(..., min_length=2)


def to_pyg_data(objects):
    positions = torch.tensor([[obj.x, obj.y] for obj in objects], dtype=torch.float32)
    object_types = torch.tensor([[obj.type] for obj in objects], dtype=torch.float32)
    speeds = torch.tensor([[obj.speed] for obj in objects], dtype=torch.float32)
    threats = torch.tensor([[obj.threat] for obj in objects], dtype=torch.float32)
    x = torch.cat([object_types, positions, speeds, threats], dim=1)

    distances = torch.cdist(positions, positions)
    edge_mask = (distances < 35) & ~torch.eye(len(objects), dtype=torch.bool)
    edge_index = edge_mask.nonzero(as_tuple=False).t().contiguous().long()
    batch = torch.zeros(len(objects), dtype=torch.long)
    return Data(x=x, edge_index=edge_index, batch=batch)


app = FastAPI(title="Tactical Drone GNN API")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)

LABELS = {0: "Yaklaş", 1: "Gözlemle", 2: "Kaçın"}


@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_PATH.name}


@app.post("/predict")
def predict(request: PredictionRequest):
    if request.objects[0].type != 0:
        raise HTTPException(422, "İlk nesne Dron (type=0) olmalıdır.")
    if request.objects[1].type != 1:
        raise HTTPException(422, "İkinci nesne Hedef (type=1) olmalıdır.")

    data = to_pyg_data(request.objects)
    with torch.inference_mode():
        probabilities = torch.softmax(model(data), dim=1)[0]
        class_id = int(probabilities.argmax().item())

    return {
        "class_id": class_id,
        "label": LABELS[class_id],
        "probabilities": [round(float(value), 6) for value in probabilities],
    }


nest_asyncio.apply()
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
ngrok.kill()
public_url = ngrok.connect(8000).public_url
server_config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning")
server = uvicorn.Server(server_config)
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()

print(f"Public URL: {public_url}")
print(f"Predict endpoint: {public_url}/predict")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.8/144.8 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 9.6 MB/s eta 0:00:00


ValueError: NGROK_AUTH_TOKEN alanına kendi ngrok tokeninizi yazın.